In [ ]:
############# TRAINING NOTEBOOK #############
# Initialization and Imports

import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from Configuration import *
import os

from models.Transformers import PoetEmbedder
from datasets.MaskedDatasetPoetry import DynamicMaskedPoetryDataset
from Tokenizers.tokenizer import BasePoetryTokenizer, WordPieceTokenizer, SyllableTokenizer

############# CONFIGURATION #############
MAX_SEQ_LEN =       TRANSFORMER_CONFIGURATION.PARAMETERS()["MAX_SEQ_LEN"]
BATCH_SIZE =        TRANSFORMER_CONFIGURATION.TRAINING_PARAMETERS()["BATCH_SIZE"]
EPOCHS =            TRANSFORMER_CONFIGURATION.TRAINING_PARAMETERS()["EPOCHS"]
LEARNING_RATE =     TRANSFORMER_CONFIGURATION.TRAINING_PARAMETERS()["LEARNING_RATE"]
TRAIN_SPLIT_RATIO = TRANSFORMER_CONFIGURATION.TRAINING_PARAMETERS()["TRAIN_SPLIT_RATIO"]
SEED =              TRANSFORMER_CONFIGURATION.TRAINING_PARAMETERS()["SEED"]
MAX_LR =            TRANSFORMER_CONFIGURATION.TRAINING_PARAMETERS()["MAX_LR"]
JSON_DIR =          PATH_CONFIGURATION.DATASETS_PATH()["POETRY"]

TRAINING_PHASE = False  # Set to True to train the model, False to skip training and load existing weights

# Set random seed for reproducibility
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")
all_poems = BasePoetryTokenizer.get_all_poems_from_directory(json_dir=JSON_DIR)


Training on device: cuda


In [ ]:
#################### TRAINING UTILITY FUNCTIONS ####################
def train_model(
    model: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    test_loader: torch.utils.data.DataLoader,
    tokenizer: BasePoetryTokenizer,
    epochs: int,
    max_lr: float,
    weights_path: str,
    device: torch.device
):
    """
    Training loop for the bidirectional transformer model using the OneCycleLR scheduler to achieve super-convergence.

    Se `weights_path` esiste, il training riprende dall'ultimo checkpoint salvato
    (epoca successiva, stato di optimizer e scheduler inclusi). Se non esiste,
    il training parte da zero.

    Parameters:
        model (nn.Module): the transformer model to train.
        train_loader (DataLoader): DataLoader for the training set.
        test_loader (DataLoader): DataLoader for the test/validation set.
        tokenizer: BasePoetryTokenizer to obtain the vocab_size.
        epochs (int): Total number of training epochs.
        max_lr (float): Maximum learning rate for the OneCycleLR scheduler.
        weights_path (str): Path where to save or load the model weights.
        device (torch.device): Computational device (CPU or CUDA).

    """
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.01)

    # OneCycleLR scheduler for super-convergence with a maximum learning rate of max_lr and
    # a warm-up phase of 10% of the total training steps
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        steps_per_epoch=len(train_loader),
        epochs=epochs,
        pct_start=0.1
    )

    vocab_size = len(tokenizer.vocab)
    model.to(device)

    start_epoch = 1
    best_test_loss = float("inf")

    if os.path.exists(weights_path):
        print(f"\n[INFO] Weights found at '{weights_path}'. Restoring state...")
        checkpoint = torch.load(weights_path, map_location="cpu")

        if "model_state_dict" in checkpoint: # If the checkpoint contains the required keys, restore the model and the checkpoint
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
            start_epoch = checkpoint.get("epoch", 0) + 1
            best_test_loss = checkpoint.get("best_test_loss", float("inf"))
        else:
            print("[WARN] Legacy checkpoint format detected (model weights only). "
                  "Optimizer/scheduler state will be reinitialized and training will restart from epoch 1.")

        model.to(device)
        print(f"[SUCCESS] Weights loaded successfully! Resuming from epoch {start_epoch}/{epochs} "
              f"(best test loss: {best_test_loss:.4f})")
    else:
        print(f"\n[INFO] No weights found at '{weights_path}'. Starting training from scratch (Max LR: {max_lr})...")

    if start_epoch > epochs:
        print(f"[INFO] Training already completed for all {epochs} epochs. Nothing to do.")
        return

    for epoch in range(start_epoch, epochs + 1):
        # ----------------- FASE DI TRAINING -----------------
        model.train()
        total_train_loss = 0.0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            padding_mask = batch["padding_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, padding_mask=padding_mask)

            loss = criterion(logits.view(-1, vocab_size), labels.view(-1))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # ----------------- FASE DI TEST / VALIDAZIONE -----------------
        model.eval()
        total_test_loss = 0.0

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                padding_mask = batch["padding_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids, padding_mask=padding_mask)
                loss = criterion(logits.view(-1, vocab_size), labels.view(-1))
                total_test_loss += loss.item()

        avg_test_loss = total_test_loss / len(test_loader)
        current_lr = scheduler.get_last_lr()[0]

        print(f"Epoch [{epoch:02d}/{epochs:02d}] | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")

        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_test_loss": best_test_loss,
            }
            torch.save(checkpoint, weights_path)
            print(f" -> Checkpoint salvato! Miglior Test Loss: {best_test_loss:.4f}")

    print(f"\n[SUCCESS] Addestramento completato! Pesi del modello salvati in '{weights_path}'.")

In [ ]:
############## SYLLABLE DATASET & SPLITTING INITIALIZATION #############

# Shuffle and split the dataset into training and testing sets
random.shuffle(all_poems)
split_idx = int(len(all_poems) * TRAIN_SPLIT_RATIO)
train_poems = all_poems[:split_idx]
test_poems = all_poems[split_idx:]

print(f"Total poems: {len(all_poems)} | Train set: {len(train_poems)} | Test set: {len(test_poems)}")



Poesie totali: 25333 | Train set: 22799 | Test set: 2534
[INFO] Pre-tokenizzazione e chunking dinamico in memoria...
[INFO] Pre-tokenizzazione e chunking dinamico in memoria...


In [ ]:
############# SYLLABLE TOKENIZER & TRANSFORMER INITIALIZATION #############
syllabizer = SyllableTokenizer.from_config()
model = PoetEmbedder.from_config(len(syllabizer.vocab)).to(device)

if os.path.exists(PATH_CONFIGURATION.MODELS_WEIGHTS_PATH()["SYLLABLE_TRANSFORMER"]):
    print(f"\n[INFO] Weights found at '{PATH_CONFIGURATION.MODELS_WEIGHTS_PATH()['SYLLABLE_TRANSFORMER']}'.")
    model.load_state_dict(torch.load(PATH_CONFIGURATION.MODELS_WEIGHTS_PATH()["SYLLABLE_TRANSFORMER"])["model_state_dict"])
    model.eval()
    print("Loaded existing model weights.")

############# DATASET CREATION #############
if TRAINING_PHASE:
    train_dataset = DynamicMaskedPoetryDataset(
        poems=train_poems, 
        tokenizer=syllabizer, 
        max_len=MAX_SEQ_LEN,
        mask_prob=0.15
    )

    test_dataset = DynamicMaskedPoetryDataset(
        poems=test_poems, 
        tokenizer=syllabizer, 
        max_len=MAX_SEQ_LEN,
        mask_prob=0.15
    )

In [ ]:
############## TRAINING AND EVALUATION LOOP #############
WEIGHTS_PATH =  PATH_CONFIGURATION.MODELS_WEIGHTS_PATH()["SYLLABLE_TRANSFORMER"]

if TRAINING_PHASE:
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=8, 
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=8, 
        pin_memory=True
    )

    train_model(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        tokenizer=syllabizer,
        epochs=EPOCHS,
        max_lr=MAX_LR,
        weights_path=WEIGHTS_PATH,
        device=device
    )
else:
    print(f"\n[INFO] Skipping Syllable Transformer training phase.")


[INFO] Trovato file di pesi salvato in './weights/syllable_transformer_weights.pt'.


RuntimeError: Error(s) in loading state_dict for PoetEmbedder:
	Missing key(s) in state_dict: "token_embedding.weight", "positional_embedding.weight", "layers.0.attention.W_q.weight", "layers.0.attention.W_k.weight", "layers.0.attention.W_v.weight", "layers.0.attention.W_o.weight", "layers.0.attention.W_o.bias", "layers.0.norm1.weight", "layers.0.norm1.bias", "layers.0.ffn.0.weight", "layers.0.ffn.0.bias", "layers.0.ffn.2.weight", "layers.0.ffn.2.bias", "layers.0.norm2.weight", "layers.0.norm2.bias", "layers.1.attention.W_q.weight", "layers.1.attention.W_k.weight", "layers.1.attention.W_v.weight", "layers.1.attention.W_o.weight", "layers.1.attention.W_o.bias", "layers.1.norm1.weight", "layers.1.norm1.bias", "layers.1.ffn.0.weight", "layers.1.ffn.0.bias", "layers.1.ffn.2.weight", "layers.1.ffn.2.bias", "layers.1.norm2.weight", "layers.1.norm2.bias", "layers.2.attention.W_q.weight", "layers.2.attention.W_k.weight", "layers.2.attention.W_v.weight", "layers.2.attention.W_o.weight", "layers

In [ ]:
############################# WORD PIECE TOKENIZER & TRANSFORMER INITIALIZATION ############################
torch.cuda.empty_cache()
tokenizer = WordPieceTokenizer.from_config()
word_transformer = PoetEmbedder.from_config(len(tokenizer.vocab)).to(device)
WEIGHTS_PATH = PATH_CONFIGURATION.MODELS_WEIGHTS_PATH()["WORDPIECE_TRANSFORMER"]

if os.path.exists(WEIGHTS_PATH):
    print(f"\n[INFO] Weights found at '{WEIGHTS_PATH}'.")
    word_transformer.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    word_transformer.eval()
    print("[SUCCESS] Weights loaded successfully!")

if TRAINING_PHASE:
    train_set = DynamicMaskedPoetryDataset(
    poems=train_poems,
    tokenizer=tokenizer,
    max_len=MAX_SEQ_LEN,
    mask_prob=0.15
    )

    test_set = DynamicMaskedPoetryDataset(
        poems=test_poems,
        tokenizer=tokenizer,
        max_len=MAX_SEQ_LEN,
        mask_prob=0.15
    )

In [ ]:
############################## WORD TRANSFORMER TRAINING ##################################
if TRAINING_PHASE:
    print(f"\n[INFO] Avvio dell'addestramento del WordPiece Transformer (Max LR: {MAX_LR})...")
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True,
        num_workers=8,
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=8,
        pin_memory=True
    )

    train_model(
        model=word_transformer,
        train_loader=train_loader,
        test_loader=test_loader,
        tokenizer=tokenizer,
        epochs=EPOCHS,
        max_lr=MAX_LR,
        weights_path=WEIGHTS_PATH,
        device=device
    )
else:
    print(f"\n[INFO] Skipping WordPiece Transformer training phase.")


[INFO] Avvio dell'addestramento del WordPiece Transformer (Max LR: 0.0005)...

[INFO] Avvio dell'addestramento con OneCycleLR (Max LR: 0.0005)...
